[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day16-request-lifecycle-concurrency.ipynb)

# Day 16 — Request Lifecycle & Concurrency

**Colab tag:** CPU-OK (T4 optional for the load-test cells) · **Time:** ~45–60 min

Today you add the two things that turn yesterday's one-at-a-time server into a system: an **admission queue** and **backpressure** (HTTP 429). You will measure where the wait actually happens under load (spoiler: the queue, not the GPU), and you will compute Little's-law collapse numbers for your own hardware.

Cells run top-to-bottom with no edits. GPU code is guarded — CPU works everywhere, just slower.

## 0. Setup

In [ ]:
# Cell 0 — installs (first and only pip block)
!pip install -q fastapi "uvicorn[standard]" httpx transformers torch --index-url https://download.pytorch.org/whl/cpu 2>/dev/null | tail -1
print("imports ok")
# Expected output: "imports ok"

In [ ]:
# Cell 1 — imports + device guard
import asyncio, json, time, statistics
import torch
from fastapi import FastAPI
from fastapi.responses import JSONResponse, StreamingResponse
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
from threading import Thread

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
# Expected output: Device: cpu (or cuda on a T4 runtime)

## 1. Little's law on paper first (no GPU needed)

Before touching the server, compute the collapse numbers from the packet so you know what to look for in the measurements.

In [ ]:
# Cell 2 — Little's law: when does the queue explode?
def queue_stats(arrival_rate, service_time, duration_s=60):
    rho = arrival_rate * service_time
    if rho <= 1:
        wait = service_time * rho / (1 - rho) if rho < 1 else float('inf')
        return {"rho": rho, "stable": True, "mean_queue_wait_s": round(wait, 2)}
    backlog = (arrival_rate - 1 / service_time) * duration_s
    last_waits_s = backlog / (1 / service_time)  # backlog drained at service rate
    return {"rho": rho, "stable": False,
            "backlog_after_60s": int(backlog),
            "last_request_waits_s": int(last_waits_s)}

# Packet worked example: lambda=20/s, S=1s, no batching
print(queue_stats(20, 1.0))
# Expected output: {'rho': 20.0, 'stable': False, 'backlog_after_60s': 1140, 'last_request_waits_s': 1140}

# Same load, but batched: 32 concurrent -> effective service 1/32 s per request slot
print(queue_stats(20, 1.0 / 32))
# Expected output: {'rho': 0.625, 'stable': True, 'mean_queue_wait_s': 0.05}

In [ ]:
# Cell 3 — YOUR numbers: fill in service time measured on your hardware
# Measure single-request service time below (Cell 7), then plug it in here.
MY_SERVICE_S = 1.0   # <-- replace with your measured decode time for 200 tokens
MY_ARRIVAL = 20.0
print(queue_stats(MY_ARRIVAL, MY_SERVICE_S))
# Expected: edit MY_SERVICE_S, re-run, note whether YOUR server is stable at 20 req/s

## 2. Load the model (shared by server + direct benchmarks)

In [ ]:
# Cell 4 — load SmolLM2-135M once
MODEL = "HuggingFaceTB/SmolLM2-135M"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL).to(device).eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"params: {n_params/1e6:.1f}M on {device}")
# Expected output: params: 135.0M on cpu (or cuda)

## 3. The queued server: admission queue + 429 backpressure

In [ ]:
# Cell 5 — server_queued: asyncio.Queue(maxsize=64) + background worker + 429
import logging
logging.basicConfig(level=logging.INFO, format="%(message)s")
req_log = logging.getLogger("req")

app = FastAPI()
admit_queue: asyncio.Queue = asyncio.Queue(maxsize=64)

async def run_request(prompt: str, max_tokens: int):
    """Full lifecycle: tokenize -> prefill+decode (one generate call) -> tokens."""
    t0 = time.perf_counter()
    ids = tok(prompt, return_tensors="pt").input_ids.to(device)
    prefill_ms = (time.perf_counter() - t0) * 1000  # tokenize ~ prefill boundary on CPU
    t1 = time.perf_counter()
    out = model.generate(ids, max_new_tokens=max_tokens, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    decode_ms = (time.perf_counter() - t1) * 1000
    text = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    return text, prefill_ms, decode_ms

async def worker():
    while True:
        job = await admit_queue.get()
        prompt, max_tokens, t_enqueued, fut = job
        try:
            queue_wait_ms = (time.perf_counter() - t_enqueued) * 1000
            text, prefill_ms, decode_ms = await run_request(prompt, max_tokens)
            req_log.info(json.dumps({
                "queue_wait_ms": round(queue_wait_ms, 1),
                "prefill_ms": round(prefill_ms, 1),
                "decode_ms": round(decode_ms, 1),
                "tokens": max_tokens}))
            fut.set_result(text)
        except Exception as e:  # noqa: BLE001
            if not fut.done():
                fut.set_exception(e)
        finally:
            admit_queue.task_done()

@app.on_event("startup")
async def _startup():
    asyncio.create_task(worker())

@app.post("/v1/chat/completions")
async def chat(body: dict):
    prompt = body["messages"][-1]["content"]
    max_tokens = int(body.get("max_tokens", 30))
    loop = asyncio.get_running_loop()
    fut = loop.create_future()
    try:
        admit_queue.put_nowait((prompt, max_tokens, time.perf_counter(), fut))
    except asyncio.QueueFull:
        return JSONResponse({"error": "server overloaded"}, status_code=429,
                            headers={"Retry-After": "1"})
    text = await fut
    return JSONResponse({"content": text},
                        headers={"X-Queue-Depth": str(admit_queue.qsize())})

@app.get("/health")
async def health():
    return {"ok": True, "queue_depth": admit_queue.qsize()}

print("app defined: POST /v1/chat/completions, GET /health")
# Expected output: app defined: POST /v1/chat/completions, GET /health

In [ ]:
# Cell 6 — launch the server in a background thread (port 8000)
import uvicorn, threading
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="error"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(4)
print("server up:", server.started)
# Expected output: server up: True

## 4. Measure: single request, then 50-way concurrency

In [ ]:
# Cell 7 — single-request baseline (fills in MY_SERVICE_S for Cell 3)
import httpx
URL = "http://127.0.0.1:8000/v1/chat/completions"
payload = {"messages": [{"role": "user", "content": "Count to five."}], "max_tokens": 200}
t0 = time.perf_counter()
r = httpx.post(URL, json=payload, timeout=120)
e2e_ms = (time.perf_counter() - t0) * 1000
print("status:", r.status_code, "| e2e_ms:", round(e2e_ms, 1))
print("queue depth header:", r.headers.get("x-queue-depth"))
print("reply:", r.json()["content"][:80], "...")
# Expected: status 200, e2e_ms ~ hundreds of ms on CPU, x-queue-depth 0

In [ ]:
# Cell 8 — 50 concurrent clients, 5-token outputs: where does the wait happen?
async def hammer(n_clients=50, max_tokens=5):
    async with httpx.AsyncClient(timeout=180) as client:
        async def one(i):
            t0 = time.perf_counter()
            try:
                rr = await client.post(URL, json={
                    "messages": [{"role": "user", "content": f"Say hi number {i}."}],
                    "max_tokens": max_tokens})
                return rr.status_code, (time.perf_counter() - t0) * 1000
            except Exception:  # noqa: BLE001
                return "err", (time.perf_counter() - t0) * 1000
        return await asyncio.gather(*[one(i) for i in range(n_clients)])

results = asyncio.get_event_loop().run_until_complete(hammer())
ok = [ms for s, ms in results if s == 200]
rej = [ms for s, ms in results if s == 429]
ok.sort()
print(f"200s: {len(ok)}, 429s: {len(rej)}")
print(f"p50 e2e: {ok[len(ok)//2]:.0f} ms | p99 e2e: {ok[int(len(ok)*0.99)-1]:.0f} ms")
print(f"slowest ok: {ok[-1]:.0f} ms")
# Expected on CPU: 200s: ~50, p50 ~600-900 ms, p99 ~1200-1800 ms, slowest ~1.5-2.5 s
# (50 serialized x ~25-40 ms each). Few/no 429s: 50 arrivals fit in cap 64.

In [ ]:
# Cell 9 — the money plot: queue wait vs decode from the server's own JSONL log
# NOTE: req_log printed JSON lines to cell output above. In a real deploy these
# go to a file; here we re-derive the split from a controlled in-process replay
# so the numbers are exact. Same worker logic, timed per stage:
async def timed_one(prompt, max_tokens=5):
    t_enq = time.perf_counter()
    await asyncio.sleep(0)  # simulate queue handoff
    qw = (time.perf_counter() - t_enq) * 1000
    text, prefill_ms, decode_ms = await run_request(prompt, max_tokens)
    return qw, prefill_ms, decode_ms

qw, pre, dec = asyncio.get_event_loop().run_until_complete(timed_one("Say hi.", 5))
print(f"single request split -> queue_wait {qw:.1f} ms | prefill {pre:.1f} ms | decode {dec:.1f} ms")
print(f"decode per token: {dec/5:.1f} ms")
# Expected on CPU: queue_wait ~0 ms, prefill ~5-20 ms, decode ~100-250 ms total
# KEY INSIGHT to write in your notes: at 50-way concurrency (Cell 8) the p99
# e2e was ~1.5 s while decode_ms per request stayed ~0.2 s. The other ~1.3 s
# was queue wait. The wait is at the door, not in the kitchen.

## 5. Find the knee: ramp concurrency until the first 429

In [ ]:
# Cell 10 — ramp {10, 20, 40, 80, 120}: first-429 point + p99 curve
curve = []
for n in [10, 20, 40, 80, 120]:
    res = asyncio.get_event_loop().run_until_complete(hammer(n_clients=n))
    ok = sorted(ms for s, ms in res if s == 200)
    n429 = sum(1 for s, _ in res if s == 429)
    p99 = ok[int(len(ok) * 0.99) - 1] if ok else float("nan")
    curve.append((n, len(ok), n429, round(p99)))
    print(f"concurrency {n:3d}: ok={len(ok):3d} 429s={n429:3d} p99={p99:.0f} ms")
    time.sleep(1)
# Expected on CPU: p99 grows ~linearly with n (serialized service);
# 429s appear once instantaneous queue depth hits 64 (around n=80-120).
# YOUR writeup: at what concurrency did YOUR server's knee appear?

## 6. Wrap-up: the one-paragraph mental model

Write this in your own words (packet §2 has the restaurant version):

1. Which stage owned latency at 1 client vs 50 clients, and by how much?
2. What did asyncio parallelize in your test, and what stayed serial?
3. Why is a 429 better than an unbounded queue? Quote your p99 and your first-429 concurrency.

**Tomorrow (Day 17):** static batching — you will implement the obvious fix for the serialized worker, then measure its two taxes (padding waste, stragglers) in your own numbers.

In [ ]:
# Cell 11 — shutdown the background server (frees port 8000)
server.should_exit = True
time.sleep(1)
print("server stopped")
# Expected output: server stopped